In [13]:
import pandas as pd

TEXT_COLS = [
    "shipment_id", "entry_no", "hs_code", "product_desc", "supplier",
    "country_origin", "country_destination", "broker", "port_of_entry",
    "incoterm", "license_required", "license_obtained", "fta_claimed",
    "restricted_party_pass", "container_number", "slow_clearance",
]
DATE_COLS = ["entry_date", "accepted_date", "release_date"]
ARTIFACT_WEIGHT = 130837.074097   # identical value on 94 rows; not a real weight

df = pd.read_csv("customs_clearance_data.csv")   # raw, never modified
print(df.shape)

(5250, 26)


In [22]:
# THE CLEANING FUNCTION

def clean_customs(df):
    df_clean = df.copy()                                            # Step 1
    print(f"Start: {len(df_clean)} rows")

    for col in TEXT_COLS:                                           # Step 2
        df_clean[col] = df_clean[col].str.strip()

    df_clean["broker"] = df_clean["broker"].str.upper()             # Step 3

    before = len(df_clean)                                          # Step 4
    df_clean = df_clean.drop_duplicates()
    print(f"Duplicates removed: {before - len(df_clean)}")

    df_clean["hs_code"] = df_clean["hs_code"].str.removesuffix("X") # Step 5

    before = len(df_clean)                                          # Step 6
    df_clean = df_clean.dropna(subset=["hs_code", "country_origin"]).reset_index(drop=True)
    print(f"Null hs_code/country_origin dropped: {before - len(df_clean)}")

        # Step 7: derive HS chapter and section (needs clean 6-digit codes, so after steps 5-6)
    section_edges = [0, 5, 14, 15, 24, 27, 38, 40, 43, 46, 49, 63, 67, 70, 71, 83, 85, 89, 92, 93, 96, 99]
    section_labels = ["I", "II", "III", "IV", "V", "VI", "VII", "VIII", "IX", "X", "XI",
                      "XII", "XIII", "XIV", "XV", "XVI", "XVII", "XVIII", "XIX", "XX", "XXI"]

    df_clean["hs_chapter"] = df_clean["hs_code"].str[:2].astype(int)
    df_clean["hs_section"] = pd.cut(df_clean["hs_chapter"], bins=section_edges, labels=section_labels)
    
    
    
    for col in DATE_COLS:                                           # Step 8
        df_clean[col] = pd.to_datetime(df_clean[col])

    df_clean.loc[df_clean["weight_kg"] == ARTIFACT_WEIGHT, "weight_kg"] = float("nan")       
    
    # Step 9: 94 weights hold an identical artifact value (~130,837 kg); nothing lies between ~4,900 and it
    before = len(df_clean)
    df_clean = df_clean[~(df_clean["weight_kg"] > 10000)].reset_index(drop=True)
    print(f"Artifact-weight rows dropped: {before - len(df_clean)}")

    assert df_clean["hs_code"].str.len().eq(6).all()
    assert df_clean["hs_code"].str.isdigit().all()
    assert df_clean["broker"].nunique() == 6
    assert df_clean[["hs_code", "country_origin", "weight_kg"]].isnull().sum().sum() == 0
    assert df_clean["shipment_id"].is_unique
    assert df_clean["hs_section"].notnull().all(), "Chapter outside 1-99"

    print(f"End: {len(df_clean)} rows")
    return df_clean

In [23]:
# RUN AND SAVE

df_clean = clean_customs(df)
df_clean.to_csv("customs_clean.csv", index=False)

Start: 5250 rows
Duplicates removed: 0
Null hs_code/country_origin dropped: 410
Artifact-weight rows dropped: 94
End: 4746 rows


In [24]:
df_clean["weight_kg"].max()    # should be under about 5,000, not 130,837

np.float64(4999.99)

In [25]:
bad = df_clean[df_clean["release_date"] < df_clean["accepted_date"]]
print((bad["accepted_date"] - bad["release_date"]).describe())
print(bad["slow_clearance"].value_counts(normalize=True))

count                       179
mean     0 days 02:14:04.692737
std      0 days 01:15:00.163828
min             0 days 01:00:00
25%             0 days 01:00:00
50%             0 days 02:00:00
75%             0 days 03:00:00
max             0 days 05:00:00
dtype: object
slow_clearance
N    0.843575
Y    0.156425
Name: proportion, dtype: float64


In [26]:
df_clean["hours_to_release"] = (df_clean["release_date"] - df_clean["entry_date"]).dt.total_seconds() / 3600
print(df_clean.groupby("slow_clearance")["hours_to_release"].describe())

                 count       mean        std  min   25%   50%    75%    max
slow_clearance                                                             
N               3438.0  23.512507  13.497992  0.0  12.0  23.0   35.0   47.0
Y               1208.0  82.039735  24.351440  0.0  64.0  83.0  102.0  120.0


In [27]:
mask = df_clean["release_date"] < df_clean["accepted_date"]
df_clean.loc[mask, "release_date"] = pd.NaT
df_clean = df_clean.drop(columns="hours_to_release", errors="ignore")
df_clean.to_csv("customs_clean.csv", index=False)
print(df_clean.shape)    # (4746, 26)

(4746, 28)


In [28]:
df_clean["duty_rate_declared"] = df_clean["duty_amount_usd"] / df_clean["declared_value_usd"]
df_clean["duty_rate_invoice"] = df_clean["duty_amount_usd"] / df_clean["invoice_value_usd"]
print(df_clean.groupby("hs_code")[["duty_rate_declared", "duty_rate_invoice"]].std().mean())

duty_rate_declared    0.009658
duty_rate_invoice     0.004623
dtype: float64


In [32]:
extras = ["chapter", "hours_to_release", "duty_rate_declared", "duty_rate_invoice"]
df_clean = df_clean.drop(columns=extras, errors="ignore")
print(df_clean.shape)        # (4746, 28)
df_clean.to_csv("customs_clean.csv", index=False)

(4746, 28)
